# Week 5 Lab: Estimating the Return to Schooling

## Learning Objectives

By the end of this lab, you will be able to:
1. Load and prepare econometric data in Julia
2. Construct the regressor matrix for a multiple regression
3. Estimate OLS coefficients using matrix algebra
4. Compute standard errors under homoskedasticity
5. Construct confidence intervals and perform hypothesis tests

One of the most often studied problems in labor economics is the **return to schooling**. In a Beckerian human capital investment model, people *choose* the level of schooling that maximizes the present value of their lifetime earnings. The main tradeoff that people face is this: more schooling gives people a higher earnings trajectory, but it also delays their entry into the labor market. For example, by studying for a PhD degree you forgo four (maybe six or eight) years of full-time work, but once you enter the labor market, you will have a relatively high earnings level.

Labor economists have proposed a simple regression based way of estimating the return to schooling. It boils down to this specification:

* **specification A**: $\qquad \log earn = \beta_0 + \beta_1 educ + u_A$

If $E(u_A \cdot educ) = 0$ then we can estimate $\beta_1$ consistently via OLS.

Specification A is a bit too simple, the canonical regression is:

* **specification B**: $\qquad \log earn = \beta_0 + \beta_1 educ + \beta_2 exper + \beta_3 expersq/100 + u_B$,

where $exper$ is work experience and $expersq$ is its square (why is it included?).

In one influential paper titled *Using Geographic Variation in College Proximity to Estimate the Return to Schooling* (find the 1993 NBER version!), David Card uses the following specification:

* **specification C**: $\qquad \log earn = \beta_0 + \beta_1 educ + \beta_2 exper + \beta_3 expersq/100 + \beta_4 black + \beta_5 south + \beta_6 smsa + \beta_7 smsa66 + \beta_8 reg661 + \cdots + \beta_{15} reg668 + u_C$ 

Card uses the *National Longitudinal Survey of Young Men* from the US. He mostly relies on data from the year 1976.

The extra variables in Card's model are:

* $black$ is a dummy if the person is African American
* $south$ is a dummy if the person is from the American South (which, on average, has a lower level of development)
* $smsa$ is a dummy if the person is from a standard metropolitan statistical area (a city) in 1976 (the time of the survey)
* $smsa66$ is a dummy if the person's location was in a SMSA in 1966
* $reg661$ to $reg668$ are regional dummies

## Your Job

Estimate Card's specification. In particular:
* estimate $\beta_0$ through $\beta_{15}$
* obtain an estimate for the asymptotic covariance (assume homoskedasticity)
* obtain standard errors for the coefficient estimates
* construct a confidence interval for the return to schooling

Compare your estimates to Card's Table 2.

## Useful Results from the Lecture

Before you start coding, please recall the results from Lecture 3. 

The OLS estimator is given by $\widehat{\beta}^\text{OLS} = (X'X)^{-1} X'Y$.

We obtained the following asymptotic distribution result:

$$\sqrt{N} (\widehat{\beta}^\text{OLS} - \beta) \overset{d}{\to} N(0, \Omega)$$

where, under homoskedasticity, the $K \times K$ matrix $\Omega$ simplifies to

$$\Omega = \sigma_u^2 \cdot E(X_i X_i')^{-1}$$

(You derived this in problem set 3).

We say that

* $\Omega$ is the asymptotic variance under homoskedasticity of $\sqrt{N} (\widehat{\beta}^\text{OLS} - \beta)$

* $\Omega/N$ is the asymptotic variance under homoskedasticity of $\widehat{\beta}^\text{OLS}$

* We take this to mean that $\widehat{\beta}^\text{OLS}$ has an approximate normal distribution with mean $\beta$ and variance $\Omega/N$

We don't know $\Omega$, but the obvious plug-in (analog) estimator is

$$\widehat{\Omega}_{homosk} = s_u^2 \cdot \left(\sum X_i X_i' / N\right)^{-1} = N \cdot s_u^2 \cdot (X'X)^{-1}$$

where $s_u^2$ is the unbiased estimator of $\sigma_u^2$ which in Lecture 3 was given by

$$s_u^2 := \frac{1}{N-k} \sum \hat{u}_i^2 = \frac{1}{N-k} \hat{u}' \hat{u}$$

It follows that a good guess for the asymptotic variance under homoskedasticity of $\widehat{\beta}^\text{OLS}$ is 
$s_u^2 \cdot (X'X)^{-1}$

<div class="alert alert-success">

**Key Result:** 

We take this to mean that $\hat{\beta}_{OLS}$ has an approximate variance of $(1/N) \cdot \widehat{\Omega}_{homosk} = s_u^2 \cdot (X'X)^{-1}$.
    
</div>

The square root of the diagonal elements will therefore be the standard errors, that is,

$$\text{se} (\hat{\beta}_{OLS}) = s_u \cdot \sqrt{\text{diag}(X'X)^{-1}}$$

(which is of dimension $K \times 1$).

---
## Julia Coding

### Typing Greek Symbols in Julia

Julia supports Unicode characters, so you can use Greek letters directly in your code. This makes the code look like the mathematical notation. In Julia (and Jupyter), type the LaTeX name followed by Tab:

| Type this | Press | Get |
|-----------|-------|-----|
| `\beta`   | Tab   | `β` |
| `\sigma`  | Tab   | `σ` |
| `\Omega`  | Tab   | `Ω` |
| `\^2`     | Tab   | `²` |

For example, to type `β_hat`, type `\beta` then press Tab, then type `_hat`.

### Reading data

Make sure to put the file `card.csv` in the same directory as your Julia notebook.

In [ ]:
# Load packages
using DelimitedFiles
using LinearAlgebra, Statistics, Printf
using Distributions   # Normal(), TDist(), quantile()

**Note on file paths:** This notebook reads data from the `datasets/` folder. The directory structure for this course is:

```
EMET8014/
├── computer_labs/
│   ├── week_5.ipynb  ← you are here
│   ├── week_6.ipynb
│   └── ...
└── datasets/
    ├── card.csv
    └── ...
```

When running code from a notebook in the `computer_labs/` folder, we use `../datasets/card.csv` to access the Card data file:
- `..` goes up one level (from `computer_labs/` to `EMET8014/`)
- `datasets/` enters the datasets folder
- `card.csv` accesses the specific file

**If you've cloned or downloaded the course repository, this should work automatically.** If you get a "file not found" error, check that you haven't moved the folders around.

In [ ]:
# Read csv-file
# this should work for most of you
data = readdlm("../datasets/card.csv", ',')

N_rows, p = size(data)
println("data size = (N_rows, p) = ", (N_rows, p))

## Exercise 1: Extracting variables

Extract the dependent variable Y (log earnings, column 33) and construct the regressor matrix X.

The regressors are:
- Constant term (intercept)
- educ (column 4)
- exper (column 32)
- expersq/100 (column 34, divided by 100)
- black (column 22)
- south (column 24)
- smsa (column 23)
- smsa66 (column 25)
- reg661,...,reg668 (columns 12 through 19)

*Hint: Use `Float64.()` to convert data to floating point, and `hcat()` to combine columns into a matrix.*

In [ ]:
# Dependent variable: log earnings (column 33)
Y = Float64.(data[:, 33])

# Sample size
N = length(Y)

# Regressors - extract each variable
const_term  = ones(N)                       # intercept
educ        = Float64.(data[:, 4])          # years of education

# YOUR CODE HERE - extract the remaining variables
exper       = nothing  # work experience
expersq_100 = nothing  # experience squared / 100
black       = nothing  # African American dummy
south       = nothing  # South dummy
smsa        = nothing  # SMSA in 1976 dummy
smsa66      = nothing  # SMSA in 1966 dummy
reg_dummies = nothing  # regional dummies (columns 12:19)

# Construct regressor matrix X using hcat()
X = nothing  # YOUR CODE HERE

N, K = size(X)
println("X size = (N, K) = ", (N, K))

## Exercise 2: OLS Estimation

Compute the OLS estimate using the formula $\hat{\beta}^{OLS} = (X'X)^{-1} X'Y$.

*Hint: Julia's backslash operator `X \ Y` computes the least-squares solution efficiently. This is equivalent to `inv(X'X) * X'Y` but more numerically stable.*

In [ ]:
# OLS estimate
# Hint: Use the backslash operator for numerical stability
β_hat = nothing  # YOUR CODE HERE

# Return to schooling is the coefficient on educ (second element)
β_educ = nothing  # YOUR CODE HERE

# Uncomment when ready:
# @printf "Estimate of return to schooling: %.4f\n" β_educ
# @printf "Interpretation: One additional year of education is associated with %.1f%% higher wages.\n" (β_educ * 100)

## Exercise 3: Residuals and Variance Estimation

Compute the residuals $\hat{u} = Y - X\hat{\beta}$ and the unbiased variance estimator $s_u^2 = \frac{1}{N-K} \hat{u}'\hat{u}$.

*Hint: Use `dot()` from LinearAlgebra to compute inner products.*

In [ ]:
# Compute residuals
u_hat = nothing  # YOUR CODE HERE

# Unbiased estimator of σ² under homoskedasticity
s2 = nothing  # YOUR CODE HERE

# Uncomment when ready:
# @printf "Unbiased estimator of σ²: %.5f\n" s2
# @printf "Estimated σ (RMSE):       %.5f\n" sqrt(s2)

## Exercise 4: Covariance Matrix and Standard Errors

Compute the estimated covariance matrix $\hat{\Omega}/N = s_u^2 \cdot (X'X)^{-1}$ and extract the standard errors.

*Hint: Use `diag()` to extract diagonal elements and `sqrt.()` for element-wise square root.*

In [ ]:
# Estimated covariance matrix (under homoskedasticity)
Omega_hat = nothing  # YOUR CODE HERE

# Standard errors are the square roots of the diagonal elements
se = nothing  # YOUR CODE HERE

# Standard error for return to schooling
se_educ = nothing  # YOUR CODE HERE

# Uncomment when ready:
# @printf "Standard error for return to schooling: %.5f\n" se_educ

## Exercise 5: 95% Confidence Interval for Return to Schooling

Construct a 95% confidence interval using the asymptotic normal approximation.

*Hint: Use `quantile(Normal(), 0.975)` to get the critical value.*

In [ ]:
# Using asymptotic (Normal) critical value
c_normal = quantile(Normal(), 0.975)   # approximately 1.96

# Construct the confidence interval
CI_lower = nothing  # YOUR CODE HERE
CI_upper = nothing  # YOUR CODE HERE

# Uncomment when ready:
# @printf "Critical value (Normal, 97.5%%): %.4f\n" c_normal
# @printf "95%% CI for return to schooling: [%.4f, %.4f]\n" CI_lower CI_upper

## Exercise 6: Hypothesis Testing

Test whether the return to schooling is statistically significant.

Compute:
1. The t-statistic: $t = \hat{\beta}_{educ} / se(\hat{\beta}_{educ})$
2. Compare to the critical value from the t-distribution with $N-K$ degrees of freedom
3. Compute the two-sided p-value

*Hint: Use `TDist(N-K)` for the t-distribution and `cdf()` for the cumulative distribution function.*

In [ ]:
# t-statistic for H₀: β_educ = 0
t_stat = nothing  # YOUR CODE HERE

# Critical value from t-distribution
t_crit = quantile(TDist(N - K), 0.975)

# Uncomment when ready:
# @printf "t-statistic for return to schooling: %.3f\n" t_stat
# @printf "Critical value (t_{%d}, two-sided 5%%): %.4f\n" (N-K) t_crit

# Decision
# if abs(t_stat) > t_crit
#     println("Decision: Reject H₀ (β_educ = 0) at 5% significance level")
# else
#     println("Decision: Cannot reject H₀ (β_educ = 0) at 5% significance level")
# end

In [ ]:
# Two-sided p-value
# Hint: Think about what area(s) of the t-distribution correspond to a two-sided test
p_value = nothing  # YOUR CODE HERE

# Uncomment when ready:
# @printf "Two-sided p-value for H₀: β_educ = 0  →  p = %.4g\n" p_value

## Exercise 7: Full Regression Results

Display a summary table of all regression results including coefficients, standard errors, t-statistics, and p-values.

In [ ]:
# Variable names for output
var_names = ["const", "educ", "exper", "expersq/100", "black", "south", 
             "smsa", "smsa66", "reg661", "reg662", "reg663", "reg664", 
             "reg665", "reg666", "reg667", "reg668"]

# Compute t-statistics and p-values for all coefficients
# Hint: generalise what you did in Exercise 6 to all K coefficients (use broadcasting)
t_stats = nothing   # YOUR CODE HERE
p_values = nothing  # YOUR CODE HERE

# Display results (uncomment when ready)
# println("="^75)
# println("OLS REGRESSION RESULTS")
# println("="^75)
# @printf "%-12s  %10s  %10s  %10s  %10s\n" "Variable" "Coef." "Std.Err." "t-stat" "p-value"
# println("-"^75)
# for j in 1:K
#     @printf "%-12s  %10.5f  %10.5f  %10.3f  %10.4f\n" var_names[j] β_hat[j] se[j] t_stats[j] p_values[j]
# end
# println("-"^75)

In [ ]:
# R-squared: measures the fraction of variance in Y explained by the model
# Hint: you need the total sum of squares and the residual sum of squares
TSS = nothing  # YOUR CODE HERE
RSS = nothing  # YOUR CODE HERE
Rsq = nothing  # YOUR CODE HERE

# Uncomment when ready:
# @printf "\nN = %d,  K = %d\n" N K
# @printf "R² = %.4f\n" Rsq
# @printf "σ_hat  = %.4f\n" sqrt(s2)

---
## Comparison with Card (1993) Table 2

Card reports the following OLS estimates in Table 2, Column 2:

| Variable | Card (1993) |
|----------|-------------|
| educ | 0.075 (0.003) |
| exper | 0.085 (0.007) |
| expersq/100 | -0.229 (0.033) |
| black | -0.199 (0.018) |
| south | -0.148 (0.026) |
| smsa | 0.136 (0.020) |

Compare your results to these values.

## Key Takeaways

1. **Return to schooling**: Each additional year of education is associated with approximately **7.5% higher wages**.

2. **Statistical significance**: The return to schooling is highly significant (t-statistic around 21, p-value approximately 0).

3. **Experience effect**: Wages increase with experience but at a diminishing rate (negative coefficient on expersq).

4. **Interpretation caveat**: These are *associations*, not necessarily causal effects. The OLS estimate may be biased due to omitted variables (e.g., ability) that are correlated with education. Card's paper addresses this using instrumental variables.